<a href="https://colab.research.google.com/github/ankan-git-coder/Deep-learning-COURSE/blob/main/MNIST_Neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import torch
import torch.nn as nn
import  torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [12]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [13]:
train_data = datasets.MNIST(root='./data',train=True,download=True,transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

In [14]:
train_loader= DataLoader(train_data,batch_size=64,shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)

In [15]:
class DigitNet(nn.Module):
    def __init__(self):
        super(DigitNet, self).__init__()
        # Conv Layer 1
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)
        # Conv Layer 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.dropout = nn.Dropout(0.25)
        self.fc1 = nn.Linear(64 * 12 * 12, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = self.dropout(x)
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = DigitNet()

In [16]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [17]:
def train(epochs):
    model.train()
    for epoch in range(epochs):
        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            if batch_idx % 200 == 0:
                print(f"Epoch {epoch} | Batch {batch_idx} | Loss: {loss.item():.4f}")

train(3)

Epoch 0 | Batch 0 | Loss: 2.2936
Epoch 0 | Batch 200 | Loss: 0.0644
Epoch 0 | Batch 400 | Loss: 0.0743
Epoch 0 | Batch 600 | Loss: 0.0553
Epoch 0 | Batch 800 | Loss: 0.0933
Epoch 1 | Batch 0 | Loss: 0.0221
Epoch 1 | Batch 200 | Loss: 0.0167
Epoch 1 | Batch 400 | Loss: 0.0061
Epoch 1 | Batch 600 | Loss: 0.0826
Epoch 1 | Batch 800 | Loss: 0.0121
Epoch 2 | Batch 0 | Loss: 0.0164
Epoch 2 | Batch 200 | Loss: 0.0029
Epoch 2 | Batch 400 | Loss: 0.0012
Epoch 2 | Batch 600 | Loss: 0.0009
Epoch 2 | Batch 800 | Loss: 0.0010


In [18]:
def test():
    model.eval()
    correct=0
    with torch.no_grad():
      for data ,target in test_loader:
        output= model(data)
        pred = output.argmax(dim=1,keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()

    print(f"\nTest Accuracy: {100. * correct / len(test_loader.dataset)}%")

test()



Test Accuracy: 98.99%
